In [1]:
%run start

Root set to: /home/bdudas/obesity_challange


In [2]:
import os
print("Training started")
os.environ["CUDA_VISIBLE_DEVICES"] = "0,1,2,3"  
print("Using GPU:", os.environ["CUDA_VISIBLE_DEVICES"])
from src.data.perturbation_data import get_loaders
from src.models.transformerVAE import TransformerVAEEncoder, TransformerVAEDecoder, Transfomer_latent_Classifier 
from src.models.vae_trainers import StateTrainer, StateTrainer_latent, PerturbationAdder, CycleConsistentPerturbationAdder
import pytorch_lightning as pl
from pytorch_lightning.callbacks import ModelCheckpoint
from lightning.pytorch.loggers import MLFlowLogger, WandbLogger
from omegaconf import OmegaConf
import torch

Training started
Using GPU: 0,1,2,3


/home/bdudas/anaconda3/envs/vcell/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
def pretrainedmodels(encoder_config,decoder_config,trainer_config,pretrained=True):
    encoder = TransformerVAEEncoder(**encoder_config)
    decoder = TransformerVAEDecoder(**decoder_config)
    classifier_config = OmegaConf.load("configs/classifier_latent.yaml")
    classifier_config.z_dim = encoder_config.z_dim
    classifier = Transfomer_latent_Classifier(**classifier_config)
    if pretrained:
        cpkt_path = "misc/best_runs/latent_reg/checkpoints/epoch=19-step=5520.ckpt"
        state_dict = torch.load(cpkt_path,weights_only=False)
        model = StateTrainer_latent(encoder,decoder,categorizer=classifier,**trainer_config)
        model.load_state_dict(state_dict["state_dict"])
        return model.encoder, model.decoder, model.categorizer
    else:
        return encoder, decoder, classifier 

In [4]:
traning_config = OmegaConf.load("configs/pert_training.yaml")
encoder_config = OmegaConf.load("configs/encoder.yaml")
decoder_config = OmegaConf.load("configs/decoder.yaml")
#classifier_config = OmegaConf.load("configs/classifier_latent.yaml")
classifier_config = OmegaConf.load("configs/classifier.yaml")
trainer_config = OmegaConf.load("configs/pert_trainer.yaml")
latent_dim = 128
decoder_config.z_dim = latent_dim
encoder_config.z_dim = latent_dim

In [5]:
modelcpkt_adipo = ModelCheckpoint(monitor="Val/AUROC_adipo",save_top_k=3,mode="max")
modelcpkt_lipo = ModelCheckpoint(monitor="Val/AUROC_lipo",save_top_k=3,mode="max")
trainloader, valloader, *_ = get_loaders("",batch_size=traning_config.batch_size)

In [6]:
encoder, decoder, classifier = pretrainedmodels(encoder_config,decoder_config,trainer_config,pretrained=True)


/home/bdudas/anaconda3/envs/vcell/lib/python3.10/site-packages/torch/nn/modules/transformer.py:392: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(


In [ ]:
trainer_config.pretrained = True

In [8]:
model = CycleConsistentPerturbationAdder(encoder,decoder,categorizer=classifier,**trainer_config)
mlfLogger = MLFlowLogger(experiment_name=traning_config.projectName,run_name = traning_config.run_name + str(traning_config.version//4)) 

/home/bdudas/anaconda3/envs/vcell/lib/python3.10/site-packages/mlflow/tracking/_tracking_service/utils.py:177: FutureWarning: The filesystem tracking backend (e.g., './mlruns') will be deprecated in February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://github.com/mlflow/mlflow/issues/18534 for more details and migration guidance. For migrating existing data, https://github.com/mlflow/mlflow-export-import can be used.
  return FileStore(store_uri, store_uri)


In [9]:
trainer = pl.Trainer(max_epochs=traning_config.max_epochs, accelerator="auto", devices="auto",logger = mlfLogger, callbacks=[modelcpkt_adipo,modelcpkt_lipo])
trainer.fit(model, train_dataloaders=trainloader, val_dataloaders=valloader)

Trainer will use only 1 of 4 GPUs because it is running inside an interactive / notebook environment. You may try to set `Trainer(devices=4)` but please note that multi-GPU inside interactive / notebook environments is considered experimental and unstable. Your mileage may vary.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1,2,3]

  | Name             | Type                         | Params | Mode  | FLOPs
----------------------------------------------------------------------------------
0 | aucMetric        | MulticlassAUROC              | 0      | train | 0    
1 | confmat          | MulticlassConfusionMatrix    | 0      | train | 0    
2 | classwise_auc    | ClasswiseWrapper             | 0      | train | 0    
3 | EntropyLoss      | CrossEntropyLoss             | 0      | train | 0    
4 | encoder          | TransformerVAEEncoder        | 19.4 M | train | 0    
5 | decoder          | TransformerVAEDecoder  

Epoch 0: 100%|██████████| 493/493 [00:45<00:00, 10.73it/s, v_num=a14e, Train/loss_recon=0.502, Train/loss_kld=598.0, Train/loss_class=3.140, Train/loss_cycle=0.487, Train/total_loss=1.050]

/home/bdudas/anaconda3/envs/vcell/lib/python3.10/site-packages/torchmetrics/utilities/prints.py:43: UserWarning: No positive samples in targets, true positive value should be meaningless. Returning zero tensor in true positive score
  warnings.warn(*args, **kwargs)


Epoch 2:   0%|          | 0/493 [00:00<?, ?it/s, v_num=a14e, Train/loss_recon=0.513, Train/loss_kld=600.0, Train/loss_class=3.080, Train/loss_cycle=0.467, Train/total_loss=1.040, Val/loss_recon=0.486, Val/loss_kld=619.0, Val/loss_class=3.860, Val/loss_cycle=0.455, Val/total_loss=1.020]          

/home/bdudas/anaconda3/envs/vcell/lib/python3.10/site-packages/torchmetrics/utilities/prints.py:43: UserWarning: No positive samples in targets, true positive value should be meaningless. Returning zero tensor in true positive score
  warnings.warn(*args, **kwargs)


...

/home/bdudas/anaconda3/envs/vcell/lib/python3.10/site-packages/torchmetrics/utilities/prints.py:43: UserWarning: No positive samples in targets, true positive value should be meaningless. Returning zero tensor in true positive score
  warnings.warn(*args, **kwargs)


Epoch 3:  17%|█▋        | 86/493 [00:10<00:51,  7.90it/s, v_num=a14e, Train/loss_recon=0.548, Train/loss_kld=603.0, Train/loss_class=3.620, Train/loss_cycle=0.500, Train/total_loss=1.120, Val/loss_recon=0.486, Val/loss_kld=618.0, Val/loss_class=3.900, Val/loss_cycle=0.457, Val/total_loss=1.020]


Detected KeyboardInterrupt, attempting graceful shutdown ...


SystemExit: 1

/home/bdudas/anaconda3/envs/vcell/lib/python3.10/site-packages/IPython/core/interactiveshell.py:3587: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)
